# Ensemble Boosting & AdaBoost Lab Notebook
### ML Lecture (Prof. Subhash Bhagat) · 22 March 2026
**Part of:** [github.com/rpaut03l/TS-01-Pvt](https://github.com/rpaut03l/TS-01-Pvt) · ML Track

**How to use:**
1. Run each cell top-to-bottom (Shift+Enter)
2. Every line has a comment explaining what it does
3. Each cell is **self-contained** — you can run any cell by itself!

---
**Topics:** Bagging vs Boosting | AdaBoost | Decision Stumps | Weight Updates | Model Selection | GMM


## Cell 1: Install & Import Everything

In [ ]:
# ============================================================
# IMPORTS — Run this cell FIRST!
# ============================================================
# If running on Colab, these are already installed.
# If running locally, do: pip install scikit-learn matplotlib numpy

import numpy as np                    # Math operations (arrays, linear algebra)
import matplotlib.pyplot as plt       # Drawing graphs and charts
from matplotlib.colors import ListedColormap  # Custom colors for plots

# sklearn = scikit-learn = the main ML library in Python
from sklearn.tree import DecisionTreeClassifier         # Single decision tree
from sklearn.ensemble import (
    BaggingClassifier,              # Bagging = parallel ensemble
    AdaBoostClassifier,             # AdaBoost = sequential boosting
    RandomForestClassifier,         # Random Forest = bagging of trees
    GradientBoostingClassifier      # GBM = boosting with residuals
)
from sklearn.linear_model import LogisticRegression     # Simple linear classifier
from sklearn.svm import SVC                              # Support Vector Machine
from sklearn.cluster import KMeans                       # Hard clustering
from sklearn.mixture import GaussianMixture              # Soft clustering (GMM)
from sklearn.datasets import make_moons, make_blobs      # Toy datasets
from sklearn.model_selection import cross_val_score, train_test_split

print("All imports successful!")
print("numpy:", np.__version__)
import sklearn; print("sklearn:", sklearn.__version__)


## Ex1: Why Bagging Works — Single Tree vs 100 Trees
> **Baby story:** One student takes one exam = might get lucky or unlucky.
> 100 students take the exam = average score is more reliable!


In [ ]:
# ============================================================
# Ex1: SINGLE TREE vs BAGGING — See the Difference!
# ============================================================

# --- Make fake data: two half-moon shapes ---
# n_samples=300: 300 data points total
# noise=0.3: add randomness so it's not perfectly clean
# random_state=42: reproducible (same data every time you run)
X, y = make_moons(n_samples=300, noise=0.3, random_state=42)

# --- Split: 80% for training, 20% for testing ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Model 1: ONE tree (no depth limit = will overfit!) ---
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)

# --- Model 2: BAGGING of 100 trees ---
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,       # 100 trees in the bag
    bootstrap=True,         # sample WITH replacement (63.2% unique)
    oob_score=True,         # free validation using out-of-bag data
    random_state=42,
    n_jobs=-1               # use all CPU cores
)
bag_clf.fit(X_train, y_train)

# --- Print scores ---
print("ACCURACY COMPARISON:")
print(f"  Single Tree: train={single_tree.score(X_train, y_train):.3f}, "
      f"test={single_tree.score(X_test, y_test):.3f}")
print(f"  Bagging:     train={bag_clf.score(X_train, y_train):.3f}, "
      f"test={bag_clf.score(X_test, y_test):.3f}, "
      f"OOB={bag_clf.oob_score_:.3f}")
print()
print("NOTICE: Single tree has ~100% train but lower test = OVERFITTING!")
print("Bagging has slightly lower train but HIGHER test = BETTER!")


In [ ]:
# --- Visualize decision boundaries ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, clf, title in zip(
    axes,
    [single_tree, bag_clf],
    ["Single Tree (overfit!)", "Bagging 100 Trees (smooth!)"]
):
    # Create grid of tiny points to color the background
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    # Color the regions
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    # Plot actual test points
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdYlBu',
               edgecolors='black', s=50)
    ax.set_title(title, fontsize=14, fontweight='bold')

plt.suptitle("Single Tree has jagged boundaries (overfit) vs Bagging is smooth",
             fontsize=11, y=0.02)
plt.tight_layout()
plt.show()


## Ex2: Bagging vs Boosting — Same Data, Different Strategy
> **Bagging** = 100 trees trained in PARALLEL on different data subsets.
> **Boosting** = trees trained ONE AFTER ANOTHER, each fixing the previous one's mistakes.


In [ ]:
# ============================================================
# Ex2: HEAD-TO-HEAD COMPARISON
# ============================================================
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)

models = {
    "Bagging (100 trees)": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=5),
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "AdaBoost (100 stumps)": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=100, learning_rate=0.5,
        random_state=42, algorithm='SAMME'
    ),
    "Gradient Boosting (100)": GradientBoostingClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        random_state=42
    ),
}

print("5-FOLD CROSS-VALIDATION COMPARISON:")
print("=" * 55)
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    bar = "#" * int(scores.mean() * 50)  # visual bar
    print(f"  {name:<30s} {scores.mean():.3f} +/- {scores.std():.3f}  {bar}")

print()
print("KEY INSIGHT: All ensemble methods beat a single model!")
print("GBM often wins because it uses deeper trees + boosting.")


## Ex3: AdaBoost From Scratch — Every Step Visible
> **Baby story:** After each quiz, the teacher writes WRONG answers in BIG letters
> on the blackboard. Next quiz, students MUST focus on the big words!
>
> This is the FULL AdaBoost algorithm implemented manually.


In [ ]:
# ============================================================
# Ex3: ADABOOST FROM SCRATCH — 3 ROUNDS, ALL STEPS SHOWN
# ============================================================

# --- Tiny dataset (same as NUMERICAL P4) ---
X_tiny = np.array([[1,2], [2,1], [3,3], [4,2], [5,4]])
y_tiny = np.array([+1, +1, -1, -1, -1])
n = len(y_tiny)

print("DATASET:")
print(f"  X = {X_tiny.tolist()}")
print(f"  y = {y_tiny.tolist()}")
print()

# --- Initialize ---
w = np.ones(n) / n  # all equal: [0.2, 0.2, 0.2, 0.2, 0.2]
stumps = []         # store each stump
alphas = []         # store each alpha

T = 3  # number of rounds

for t in range(T):
    print(f"{'='*60}")
    print(f"ROUND {t+1}")
    print(f"{'='*60}")
    print(f"  Current weights: {np.round(w, 4)}")
    
    # --- Train a stump (using sklearn, but weighted!) ---
    stump = DecisionTreeClassifier(max_depth=1)
    stump.fit(X_tiny, y_tiny, sample_weight=w)
    preds = stump.predict(X_tiny)
    
    # --- Who got it wrong? ---
    wrong = (preds != y_tiny)
    print(f"  Predictions:     {preds.tolist()}")
    print(f"  Actual:          {y_tiny.tolist()}")
    print(f"  Wrong points:    {list(np.where(wrong)[0] + 1)}")
    
    # --- Weighted error ---
    r = np.sum(w[wrong]) / np.sum(w)
    r = np.clip(r, 1e-10, 1 - 1e-10)  # safety: avoid log(0)
    print(f"  Weighted error:  r = {r:.4f}")
    
    # --- Amount of say ---
    alpha = 0.5 * np.log((1 - r) / r)
    print(f"  Amount of say:   alpha = {alpha:.4f}")
    
    # --- Update weights ---
    # Wrong: multiply by exp(+alpha) = gets BIGGER
    # Right: multiply by exp(-alpha) = gets SMALLER
    multipliers = np.where(wrong, np.exp(+alpha), np.exp(-alpha))
    w = w * multipliers
    w = w / np.sum(w)  # normalize to sum to 1
    print(f"  New weights:     {np.round(w, 4)}")
    
    stumps.append(stump)
    alphas.append(alpha)
    print()

# --- Final classifier ---
print("=" * 60)
print("FINAL CLASSIFIER")
print("=" * 60)
print(f"  H(x) = sign( ", end="")
for t in range(T):
    sign = " + " if t > 0 else ""
    print(f"{sign}{alphas[t]:.3f} * h{t+1}(x)", end="")
print(" )")

# --- Test on each training point ---
print()
print("FINAL PREDICTIONS ON TRAINING DATA:")
for i in range(n):
    score = sum(alphas[t] * stumps[t].predict(X_tiny[i:i+1])[0]
                for t in range(T))
    pred = +1 if score > 0 else -1
    status = "CORRECT" if pred == y_tiny[i] else "WRONG"
    print(f"  x={X_tiny[i]}, score={score:+.3f}, pred={pred:+d}, "
          f"actual={y_tiny[i]:+d}  {status}")


## Ex4: Build a Decision Stump From Scratch
> **A stump** = simplest tree = one question, two answers.
> Like asking: "Is chest pain present? Yes -> sick, No -> healthy."


In [ ]:
# ============================================================
# Ex4: FIND THE BEST STUMP — Try every feature + threshold
# ============================================================

def find_best_stump(X, y, weights):
    """
    Try every possible single-feature split.
    Return the one with the lowest WEIGHTED error.
    """
    n_samples, n_features = X.shape
    best_error = float('inf')
    best_info = {}
    
    for feat in range(n_features):
        values = np.sort(np.unique(X[:, feat]))
        for i in range(len(values) - 1):
            thresh = (values[i] + values[i+1]) / 2
            for direction in [+1, -1]:
                preds = np.where(X[:, feat] <= thresh, direction, -direction)
                wrong = (preds != y)
                err = np.sum(weights[wrong])
                if err < best_error:
                    best_error = err
                    best_info = {
                        'feature': feat,
                        'threshold': thresh,
                        'direction': direction,
                        'error': err,
                        'preds': preds.copy()
                    }
    return best_info

# --- Test on chest pain data ---
# Features: [chest_pain (0=No,1=Yes), weight_kg]
X_chest = np.array([
    [1, 205], [0, 180], [1, 210], [1, 167],
    [0, 156], [0, 125], [1, 220], [0, 145]
])
y_chest = np.array([+1, +1, +1, +1, -1, -1, -1, -1])
w_chest = np.ones(8) / 8

result = find_best_stump(X_chest, y_chest, w_chest)
alpha = 0.5 * np.log((1 - result['error']) / result['error'])

print("CHEST PAIN DATASET — BEST STUMP:")
print(f"  Feature: x{result['feature']+1} "
      f"({'Chest Pain' if result['feature']==0 else 'Weight'})")
print(f"  Threshold: {result['threshold']}")
print(f"  Direction: <= thresh -> {result['direction']:+d}")
print(f"  Weighted error: {result['error']:.3f}")
print(f"  Amount of say (alpha): {alpha:.3f}")
print(f"  Predictions: {result['preds'].tolist()}")
print(f"  Actual:      {y_chest.tolist()}")


## Ex5: Watch Weights Change Each Round
> Misclassified points get HEAVIER (go UP).
> Correctly classified points get LIGHTER (go DOWN).


In [ ]:
# ============================================================
# Ex5: WEIGHT EVOLUTION PLOT — See AdaBoost "adapting"
# ============================================================
X_tiny = np.array([[1,2], [2,1], [3,3], [4,2], [5,4]])
y_tiny = np.array([+1, +1, -1, -1, -1])
n = len(y_tiny)

w = np.ones(n) / n
history = [w.copy()]

for t in range(5):  # 5 rounds
    stump = DecisionTreeClassifier(max_depth=1)
    stump.fit(X_tiny, y_tiny, sample_weight=w)
    preds = stump.predict(X_tiny)
    wrong = (preds != y_tiny)
    r = np.clip(np.sum(w[wrong]) / np.sum(w), 1e-10, 1-1e-10)
    alpha = 0.5 * np.log((1-r)/r)
    w = w * np.exp(alpha * np.where(wrong, +1, -1))
    w = w / np.sum(w)
    history.append(w.copy())

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
history = np.array(history)
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
for i in range(n):
    ax.plot(history[:, i], 'o-', color=colors[i], linewidth=2,
            markersize=8, label=f"P{i+1} (y={y_tiny[i]:+d})")

ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Sample Weight", fontsize=12)
ax.set_title("AdaBoost: Weights Change Every Round\n"
             "(Misclassified points go UP, correct ones go DOWN)",
             fontsize=13)
ax.set_xticks(range(len(history)))
ax.set_xticklabels(["Init"] + [f"R{t+1}" for t in range(5)])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Ex6: Model Selection — No Free Lunch!
> The lecture discussed how to pick the RIGHT model for your problem.
> Let's try 5 different algorithms on the same data and compare!


In [ ]:
# ============================================================
# Ex6: COMPARE 5 MODELS — Which one wins?
# ============================================================
X, y = make_moons(n_samples=500, noise=0.25, random_state=42)

models = {
    "Logistic Regression":   LogisticRegression(),
    "Decision Tree":         DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":         RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost":              AdaBoostClassifier(
                                estimator=DecisionTreeClassifier(max_depth=1),
                                n_estimators=100, random_state=42, algorithm='SAMME'),
    "SVM (RBF)":             SVC(kernel='rbf', gamma='scale'),
}

print("MODEL COMPARISON (5-fold cross-validation):")
print("=" * 60)
results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    results[name] = scores.mean()
    bar = "=" * int(scores.mean() * 40)
    print(f"  {name:<25s} {scores.mean():.3f} +/- {scores.std():.3f}  |{bar}|")

winner = max(results, key=results.get)
print(f"\nWINNER: {winner} ({results[winner]:.3f})")
print(f"\nRemember: NO FREE LUNCH! This winner is only for THIS dataset.")
print(f"Different data might have a different best model.")


## Ex7: GMM vs K-Means — Hard vs Soft Clustering
> **K-Means:** "You belong to Cluster A. Period. No questions."
> **GMM:** "You're 70% Cluster A, 30% Cluster B. Here are the probabilities!"
>
> GMM is the soft, gentle version of K-Means!


In [ ]:
# ============================================================
# Ex7: GMM vs K-MEANS COMPARISON
# ============================================================

# --- Create 3 overlapping clusters ---
X_blobs, y_true = make_blobs(
    n_samples=300,
    centers=[[0,0], [3,3], [6,0]],
    cluster_std=[1.2, 1.5, 1.0],
    random_state=42
)

# --- K-Means (hard labels) ---
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X_blobs)

# --- GMM (soft probabilities!) ---
gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(X_blobs)
gmm_labels = gmm.predict(X_blobs)
gmm_probs = gmm.predict_proba(X_blobs)  # <-- THIS is the magic!

# --- Show soft probabilities for first 8 points ---
print("GMM SOFT PROBABILITIES (first 8 points):")
print(f"  {'Point':<6} {'P(C0)':>7} {'P(C1)':>7} {'P(C2)':>7}  {'Assigned':>8}")
print("  " + "-" * 45)
for i in range(8):
    p = gmm_probs[i]
    c = np.argmax(p)
    conf = "HIGH" if p[c] > 0.9 else "MED" if p[c] > 0.7 else "LOW"
    print(f"  x[{i}]  {p[0]:>7.3f} {p[1]:>7.3f} {p[2]:>7.3f}  -> C{c} ({conf})")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_blobs[:,0], X_blobs[:,1], c=km_labels, cmap='viridis',
                s=30, alpha=0.7)
axes[0].scatter(kmeans.cluster_centers_[:,0], kmeans.cluster_centers_[:,1],
                c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[0].set_title("K-Means (Hard: 100% confidence always)", fontsize=13)

# GMM: opacity = max probability (uncertain points are faded)
max_probs = gmm_probs.max(axis=1)
axes[1].scatter(X_blobs[:,0], X_blobs[:,1], c=gmm_labels, cmap='viridis',
                s=30, alpha=max_probs)
axes[1].scatter(gmm.means_[:,0], gmm.means_[:,1],
                c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[1].set_title("GMM (Soft: faded = uncertain points)", fontsize=13)

plt.tight_layout()
plt.show()

print("\nNOTICE: In the GMM plot, FADED points are ones the model is")
print("uncertain about. K-Means can't show this uncertainty!")


In [ ]:
# --- GMM learned parameters ---
print("GMM LEARNED PARAMETERS:")
print("=" * 50)
for k in range(3):
    print(f"\nComponent {k}:")
    print(f"  Mean (center):    {gmm.means_[k].round(3)}")
    print(f"  Weight (mixing):  {gmm.weights_[k]:.3f}")
    print(f"  Covariance matrix:")
    cov = gmm.covariances_[k]
    print(f"    [{cov[0,0]:7.3f}  {cov[0,1]:7.3f}]")
    print(f"    [{cov[1,0]:7.3f}  {cov[1,1]:7.3f}]")

print("\nRECAP:")
print("  - Each component has a mean (center), covariance (shape),")
print("    and weight (how many points belong to it).")
print("  - Weights sum to 1.0 (like probabilities!).")
print("  - Next lecture: HOW GMM learns these (EM algorithm).")


## Summary & What's Next

### What We Learned Today:
1. **Ensembles** combine many weak models into one strong model
2. **Bagging** = parallel, reduces variance (e.g., Random Forest)
3. **Boosting** = sequential, reduces bias (e.g., AdaBoost, GBM)
4. **AdaBoost** uses decision stumps, adjusts weights after each round
5. **Amount of say** (alpha) depends on error: low error = high trust
6. **GMM** = soft version of K-Means, gives probabilities

### Coming Next:
- GMM deep dive (EM algorithm step by step)
- SVM and kernel methods
- Practice on real datasets!

---
*AI . ML . github.com/rpaut03l/TS-01-Pvt*
